# Coastal flood step 01: intersections

Runs input-path fixes and vector-raster intersections. Keep `run_intersections` as needed.


In [ ]:
import geopandas
import pandas
import subprocess
from pathlib import Path
import sys
import shutil
import fiona
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from pyproj import CRS

# !{sys.executable} -m pip install "nismod-snail==0.5.3"

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))


In [ ]:
output_path = base_path / "dphil_paper_3/results/01_hazard_infrastructure_network_intersections/coastal_flood_network_intersections"
data_root = base_path / "dphil_common_cross_cutting/common_incoming_data"
jamaica_metric_grid_crs = "EPSG:3448"
network_layers_csv = data_root / "networks/network_layers_hazard_intersections_details.csv"
coastal_rasters_csv = base_path / "dphil_paper_3/inputs/coastal_flood_rasters.csv"


In [ ]:
results_directory = Path(output_path)
results_directory.mkdir(parents=True, exist_ok=True)

target_crs = CRS.from_user_input(jamaica_metric_grid_crs)

reprojected_inputs_directory = results_directory / "inputs_reprojected_jamaica_crs"
reprojected_networks_directory = reprojected_inputs_directory / "networks"
reprojected_hazards_directory = reprojected_inputs_directory / "hazards"
reprojected_networks_directory.mkdir(parents=True, exist_ok=True)
reprojected_hazards_directory.mkdir(parents=True, exist_ok=True)

network_layers_table = pandas.read_csv(network_layers_csv)
network_layers_table = network_layers_table[["path"]].drop_duplicates().reset_index(drop=True)
network_layers_table["path"] = network_layers_table["path"].str.replace(
    r"^networks/", "networks/networks/", regex=True
)

coastal_rasters_table = pandas.read_csv(coastal_rasters_csv)
coastal_rasters_table["fname"] = coastal_rasters_table["path"]
coastal_rasters_table["hazard"] = "coastal"

reprojected_network_rows = []
for network_path_value in network_layers_table["path"].dropna().astype(str).unique():
    source_network_path = Path(network_path_value)
    if not source_network_path.is_absolute():
        source_network_path = data_root / source_network_path

    if network_path_value.startswith("networks/networks/"):
        relative_network_path = Path(network_path_value).relative_to("networks/networks")
    elif network_path_value.startswith("networks/"):
        relative_network_path = Path(network_path_value).relative_to("networks")
    else:
        relative_network_path = Path(network_path_value).name

    output_network_path = reprojected_networks_directory / relative_network_path
    output_network_path.parent.mkdir(parents=True, exist_ok=True)

    if output_network_path.exists():
        output_network_path.unlink()

    layer_names = fiona.listlayers(source_network_path)
    for layer_index, layer_name in enumerate(layer_names):
        network_layer = geopandas.read_file(source_network_path, layer=layer_name)
        if network_layer.crs is None:
            raise ValueError(f"Missing CRS for {source_network_path} layer {layer_name}")

        network_layer_crs = CRS.from_user_input(network_layer.crs)
        if not network_layer_crs.equals(target_crs):
            network_layer = network_layer.to_crs(target_crs)

        write_mode = "w" if layer_index == 0 else "a"
        network_layer.to_file(output_network_path, layer=layer_name, driver="GPKG", mode=write_mode)

    reprojected_network_rows.append({"path": str(output_network_path)})

network_layers_output_table = pandas.DataFrame(reprojected_network_rows).drop_duplicates().reset_index(drop=True)

reprojected_hazard_map = {}
for hazard_path_value in coastal_rasters_table["path"].dropna().astype(str).unique():
    source_hazard_path = Path(hazard_path_value)
    if not source_hazard_path.is_absolute():
        source_hazard_path = data_root / source_hazard_path

    if hazard_path_value.startswith("hazards/"):
        relative_hazard_path = Path(hazard_path_value).relative_to("hazards")
    else:
        relative_hazard_path = Path(hazard_path_value).name

    output_hazard_path = reprojected_hazards_directory / relative_hazard_path
    output_hazard_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(source_hazard_path) as source_hazard_dataset:
        if source_hazard_dataset.crs is None:
            raise ValueError(f"Missing CRS for hazard raster {source_hazard_path}")

        source_hazard_crs = CRS.from_user_input(source_hazard_dataset.crs)
        if source_hazard_crs.equals(target_crs):
            shutil.copy2(source_hazard_path, output_hazard_path)
        else:
            destination_transform, destination_width, destination_height = calculate_default_transform(
                source_hazard_dataset.crs,
                target_crs,
                source_hazard_dataset.width,
                source_hazard_dataset.height,
                *source_hazard_dataset.bounds,
            )

            destination_profile = source_hazard_dataset.profile.copy()
            destination_profile.update(
                crs=target_crs,
                transform=destination_transform,
                width=destination_width,
                height=destination_height,
            )

            with rasterio.open(output_hazard_path, "w", **destination_profile) as output_hazard_dataset:
                for band_index in range(1, source_hazard_dataset.count + 1):
                    reproject(
                        source=rasterio.band(source_hazard_dataset, band_index),
                        destination=rasterio.band(output_hazard_dataset, band_index),
                        src_transform=source_hazard_dataset.transform,
                        src_crs=source_hazard_dataset.crs,
                        dst_transform=destination_transform,
                        dst_crs=target_crs,
                        resampling=Resampling.nearest,
                    )

    reprojected_hazard_map[hazard_path_value] = str(output_hazard_path)

coastal_rasters_output_table = coastal_rasters_table.copy()
coastal_rasters_output_table["path"] = coastal_rasters_output_table["path"].map(reprojected_hazard_map)
coastal_rasters_output_table["fname"] = coastal_rasters_output_table["path"]

network_layers_output_file = results_directory / "network_layers_for_intersections.csv"
network_layers_output_table.to_csv(network_layers_output_file, index=False)

hazard_layers_output_file = results_directory / "coastal_flood_rasters_for_intersections.csv"
coastal_rasters_output_table.to_csv(hazard_layers_output_file, index=False)

vector_details_csv = network_layers_output_file
raster_details_csv = hazard_layers_output_file

CHECK_CRS_BEFORE_INTERSECTION = True
ENFORCE_JAMAICA_CRS = False  # set True to stop run if any reprojected input is missing CRS or not in jamaica_metric_grid_crs

if CHECK_CRS_BEFORE_INTERSECTION:
    crs_rows = []

    for network_path_value in network_layers_output_table["path"].dropna().astype(str).unique():
        network_path = Path(network_path_value)
        if not network_path.exists():
            crs_rows.append({
                'input_type': 'network',
                'path': network_path_value,
                'layer': None,
                'exists': False,
                'crs': None,
                'matches_jamaica_crs': False,
                'note': 'missing file',
            })
            continue

        layer_names = fiona.listlayers(network_path)
        for layer_name in layer_names:
            with fiona.open(network_path, layer=layer_name) as source_layer:
                layer_crs = source_layer.crs_wkt or source_layer.crs

            try:
                matches_crs = CRS.from_user_input(layer_crs).equals(target_crs)
            except Exception:
                matches_crs = False

            crs_rows.append({
                'input_type': 'network',
                'path': network_path_value,
                'layer': layer_name,
                'exists': True,
                'crs': str(layer_crs),
                'matches_jamaica_crs': matches_crs,
                'note': '',
            })

    for hazard_path_value in coastal_rasters_output_table["path"].dropna().astype(str).unique():
        hazard_path = Path(hazard_path_value)
        if not hazard_path.exists():
            crs_rows.append({
                'input_type': 'hazard',
                'path': hazard_path_value,
                'layer': None,
                'exists': False,
                'crs': None,
                'matches_jamaica_crs': False,
                'note': 'missing file',
            })
            continue

        with rasterio.open(hazard_path) as hazard_dataset:
            hazard_crs = hazard_dataset.crs

        try:
            matches_crs = CRS.from_user_input(hazard_crs).equals(target_crs)
        except Exception:
            matches_crs = False

        crs_rows.append({
            'input_type': 'hazard',
            'path': hazard_path_value,
            'layer': None,
            'exists': True,
            'crs': str(hazard_crs),
            'matches_jamaica_crs': matches_crs,
            'note': '',
        })

    crs_check = pandas.DataFrame(crs_rows)
    crs_check_file = results_directory / 'input_crs_check_coastal_flood.csv'
    crs_check.to_csv(crs_check_file, index=False)

    n_total = int(len(crs_check))
    n_missing = int((~crs_check['exists']).sum())
    n_mismatch = int(((crs_check['exists']) & (~crs_check['matches_jamaica_crs'])).sum())

    print("Network layers file:", vector_details_csv)
    print("Hazard layers file:", raster_details_csv)
    print("Reprojected network files:", len(network_layers_output_table))
    print("Reprojected hazard files:", coastal_rasters_output_table["path"].nunique())
    print(f"Expected CRS: {target_crs.to_string()}")
    print(f"Total inputs checked: {n_total}")
    print(f"Missing inputs: {n_missing}")
    print(f"Existing inputs not in expected CRS: {n_mismatch}")
    print('Saved:', crs_check_file)

    if n_missing > 0 or n_mismatch > 0:
        print('Non-matching inputs (first 25 rows):')
        display(crs_check.loc[(~crs_check['exists']) | (~crs_check['matches_jamaica_crs'])].head(25))

    if ENFORCE_JAMAICA_CRS and (n_missing > 0 or n_mismatch > 0):
        raise ValueError(
            'CRS check failed before intersections. Set ENFORCE_JAMAICA_CRS=False to run anyway, '
            'or reproject inputs to jamaica_metric_grid_crs first.'
        )


In [ ]:
run_intersections = True  # Set to True is you want to run this process
if run_intersections is True:
    args = [
            "python",
            str(base_path / "robyns_libraries/vector_raster_intersections.py"),
            f"{vector_details_csv}",
            f"{raster_details_csv}",
            f"{output_path}"
            ]
    print ("* Start the processing of vector-raster intersections")
    print (args)
    subprocess.run(args)
print ("* Done with the processing of vector-raster intersections")
